# Equipo 19 · Avance 6 · Revisión: pruebas separadas del pipeline

Atiende las notas de la revisión con el profesor sobre el código final del entrenamiento end-to-end. Cada nota es una
prueba independiente que se puede correr sola, sobre los datos y modelos que ya dejó el Avance 5 en Drive:

| Nota | Prueba | Sección |
|---|---|---|
| 4 | **Fuga de datos:** ¿los conjuntos de Endo-STTN y del estimador comparten videos, segmentos o frames? | 3 |
| 3 | **Estimador solo:** AF-SfMLearner tal cual (pesos SCARED), sin limpiar ni ajustar | 4 |
| 2 | **Endo-STTN solo:** se entrena solo el inpainting, con cada máscara y sin profundidad; inferencia y calidad del relleno | 5 |
| 1 | **Prueba de profundidad:** todos los modelos contra la profundidad real, a partir de sus pesos guardados | 6 |
| 5 | **Métricas con intervalos de confianza del 95 %:** AbsRel, Sq Rel, RMSE, RMSE log y δ < 1.25 | 6 |

La fuga de datos va primero: si la auditoría encuentra un cruce, las demás pruebas no serían válidas.

Este notebook **no modifica** nada de lo que produjo el Avance 5. Lee sus datos y modelos y guarda todo lo nuevo en
`MyDrive/Equipo19/avance6_revision/`.

## 0. Configuración

**Colab Pro:** *Entorno de ejecución → Cambiar tipo de entorno de ejecución →* **A100 GPU** (o L4) y **RAM alta**.
Después: *Entorno de ejecución → Ejecutar todas*.

Lo único que hay que revisar son las rutas de la celda siguiente:
- `CARPETAS_PRINCIPALES`: dónde quedaron los modelos C, D y E. La semilla 0 está en `end_to_end_mascaras/modelos`; si
  las semillas 1 y 2 se guardaron en otra carpeta, agrégala a la lista.
- `CONTROLES`: las carpetas de los controles con ancla y con Endo-STTN congelado. Si una no existe, se omite.
- `PESOS_ENDO`: los Endo-STTN reentrenados del Avance 4, los mismos que usó el Avance 5.

In [ ]:
from pathlib import Path
import os, sys, json, random, re, shutil, subprocess, time, zipfile, hashlib

# ---------- Ajusta aquí ----------
MYDRIVE = Path("/content/drive/MyDrive")
CARPETA_RAW = MYDRIVE / "Equipo19" / "raw"                       # ZIPs de C3VDv2
SALIDA = MYDRIVE / "Equipo19" / "end_to_end_mascaras"           # datos y modelos del Avance 5 (solo se leen)
SALIDA_PRUEBAS = MYDRIVE / "Equipo19" / "avance6_revision"       # todo lo nuevo de este notebook
PESOS_ENDO = {                                                   # Endo-STTN reentrenado con cada máscara (Avance 4)
    "meslouhi": MYDRIVE / "endo_corrida4" / "modelos" / "gen_mejor.pth",
    "arnold": MYDRIVE / "Equipo19" / "re_entrenar_endo" / "arnold_variaciones" / "A1" / "modelos" / "gen_mejor.pth",
}
# Carpetas donde buscar C_s<semilla>, D_<máscara>_s<semilla> y E_<máscara>_s<semilla>, en este orden
CARPETAS_PRINCIPALES = [SALIDA / "modelos"]   # agrega aquí la carpeta de las semillas 1 y 2 si es otra
CONTROLES = {                                  # nombre -> carpeta con E_meslouhi_s<semilla>
    "E · Meslouhi con ancla": MYDRIVE / "Equipo19" / "end_to_end_ancla05" / "modelos",
    "E · Meslouhi, STTN congelado": MYDRIVE / "Equipo19" / "end_to_end_sttn_congelado" / "modelos",
}
SEMILLAS = [0, 1, 2]

# La misma división y preparación del Avance 5 (se comprueba contra preparacion.json)
DIVISION = {
    "entrenamiento": ["c2_cecum_t1_v1", "c2_rectum_t1_v1", "c2_transverse1_t1_v1", "c1_ascending_t3_v1"],
    "validacion": ["c2_sigmoid_t3_v2"],
    "prueba": ["c2_descending_t4_v2", "c2_descending_t4_v3"],
}
PASO, MAX_FRAMES = 5, 300
# División con la que se reentrenó Endo-STTN (Avance 4). Se usa solo si su config.json no está junto a los pesos
DIVISION_ENDO_RESPALDO = DIVISION
PASO_ENDO = 1                 # el reentrenamiento de Endo-STTN usó todos los frames

# Nota 2: Endo-STTN solo (mismo entrenamiento que E, sin la pérdida de profundidad)
EPOCAS_ENDO = 15
LOTE_ENDO = 2                 # clips de 5 frames, como E
LR_STTN = 1e-5
INCLUIR_ORIGINAL = True       # comparar también contra el Endo-STTN publicado (gen_00009.pth)

# Nota 5: intervalos de confianza
N_BOOTSTRAP = 10000
LARGO_BLOQUE = 5              # frames consecutivos por bloque (respeta la correlación entre frames vecinos)
BLOQUE = 60                   # frames por bloque al limpiar un video con Endo-STTN
# ---------------------------------

assert "google.colab" in sys.modules, "Este notebook es para Google Colab"
from google.colab import drive
drive.mount("/content/drive")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown", "tabulate", "scikit-image"], check=True)

LOCAL = Path("/content")
EXTERNOS = LOCAL / "externos"
DATOS = LOCAL / "datos_e2e"
RES = SALIDA_PRUEBAS / "resultados"
for d in (EXTERNOS, DATOS, RES, SALIDA_PRUEBAS / "modelos"):
    d.mkdir(parents=True, exist_ok=True)

REPOS = {"Endo-STTN": ("https://github.com/endomapper/Endo-STTN.git", "9419a6c"),
         "AF-SfMLearner": ("https://github.com/ShuweiShao/AF-SfMLearner.git", "53d01a4")}
for nombre, (url, commit) in REPOS.items():
    if not (EXTERNOS / nombre).exists():
        subprocess.run(["git", "clone", "-q", url, str(EXTERNOS / nombre)], check=True)
        subprocess.run(["git", "-C", str(EXTERNOS / nombre), "checkout", "-q", commit], check=True)
    if str(EXTERNOS / nombre) not in sys.path:
        sys.path.append(str(EXTERNOS / nombre))

import gdown
PESOS_SCARED = EXTERNOS / "pesos_scared"
if not (PESOS_SCARED / "encoder.pth").exists():
    z = MYDRIVE / "Equipo19" / "Model_trained_end_to_end.zip"
    if not z.exists():
        z = EXTERNOS / "Model_trained_end_to_end.zip"
        gdown.download(id="1ZogZ5nQsfLhFAQGG_UE9A5R_osypLZf1", output=str(z), quiet=True)
    shutil.unpack_archive(str(z), str(EXTERNOS / "pesos_zip"))
    origen = next((EXTERNOS / "pesos_zip").rglob("encoder.pth")).parent
    PESOS_SCARED.mkdir(exist_ok=True)
    for f in ("encoder.pth", "depth.pth"):
        shutil.copy(origen / f, PESOS_SCARED / f)

PESOS_ORIGINAL = EXTERNOS / "gen_00009.pth"   # Endo-STTN publicado, solo como referencia en la nota 2
if INCLUIR_ORIGINAL and not PESOS_ORIGINAL.exists():
    copia = MYDRIVE / "Equipo19" / "gen_00009.pth"
    if copia.exists():
        shutil.copy(copia, PESOS_ORIGINAL)
    else:
        for file_id in ["1Evx0YSlb7YK0WWok862gddT62gDgv6ux", "14sdaDejsxgRuzHBSuqH2xEpbxuqyWI-R"]:
            try:
                if gdown.download(id=file_id, output=str(PESOS_ORIGINAL), quiet=True) and PESOS_ORIGINAL.exists():
                    break
            except Exception as e:
                print("No se pudo descargar gen_00009.pth:", type(e).__name__)
INCLUIR_ORIGINAL = INCLUIR_ORIGINAL and PESOS_ORIGINAL.exists()

for m, p in PESOS_ENDO.items():
    print(f"Endo-STTN {m:9s}:", p, "" if p.exists() else "  <- NO ENCONTRADO")
assert all(p.exists() for p in PESOS_ENDO.values()), "Faltan los Endo-STTN del Avance 4 (ver PESOS_ENDO)"
print("Endo-STTN original:", PESOS_ORIGINAL if INCLUIR_ORIGINAL else "no disponible (se omite)")
print("Resultados en     :", SALIDA_PRUEBAS)

## 1. Herramientas

Las cuatro celdas siguientes son **las mismas del notebook del Avance 5**, copiadas sin cambios, para que las pruebas
usen exactamente el código con el que se entrenó: lectura de C3VDv2 y máscaras, AF-SfMLearner, Endo-STTN y manejo de
datos.

In [ ]:
import cv2
import numpy as np

ANCHO, ALTO = 320, 256       # entrada de AF-SfMLearner (C3VD es 5:4, igual que 320x256)
LADO = 288                   # entrada de Endo-STTN
PROF_MAX_MM = 100.0


def indice(nombre):
    """Número de frame dentro del nombre de archivo ('0042_color.png' -> 42)."""
    m = re.search(r"(\d+)", Path(nombre).stem)
    return int(m.group(1)) if m else None


def listar_pares(zf):
    """Pares (indice, ruta_rgb, ruta_depth) presentes en las dos carpetas, ordenados."""
    nombres = zf.namelist()
    rgb = {indice(n): n for n in nombres if n.startswith("rgb/") and n.lower().endswith((".png", ".jpg"))}
    dep = {indice(n): n for n in nombres if n.startswith("depth/") and n.lower().endswith((".tiff", ".tif", ".png"))}
    if not rgb or not dep:
        raise ValueError(f"El ZIP no tiene rgb/ y depth/ (carpetas: {sorted({n.split('/')[0] for n in nombres})})")
    return [(i, rgb[i], dep[i]) for i in sorted(set(rgb) & set(dep) - {None})]


def leer_rgb(zf, ruta):
    return cv2.cvtColor(cv2.imdecode(np.frombuffer(zf.read(ruta), np.uint8), cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)


def leer_profundidad(zf, ruta):
    """Profundidad en mm (float32). 0 = sin dato (incluye píxeles saturados)."""
    d = cv2.imdecode(np.frombuffer(zf.read(ruta), np.uint8), cv2.IMREAD_UNCHANGED)
    d = d[..., 0] if d.ndim == 3 else d
    if d.dtype == np.uint16:
        mm = d.astype(np.float32) * (PROF_MAX_MM / 65535.0)
        mm[d == 65535] = 0
        return mm
    return d.astype(np.float32)


def _srgb_a_xyz(img):
    lin = np.where(img <= 0.04045, img / 12.92, ((img + 0.055) / 1.055) ** 2.4)
    m = np.array([[0.4124564, 0.3575761, 0.1804375],
                  [0.2126729, 0.7151522, 0.0721750],
                  [0.0193339, 0.1191920, 0.9503041]])
    return lin @ m.T


def _dilatar_1px(m):
    return cv2.dilate(m.astype(np.uint8), np.array([[0, 1, 0], [1, 1, 1], [0, 1, 0]], np.uint8)).astype(bool)


def mascara_meslouhi(rgb):
    """El Meslouhi et al. (2011), la de los autores de Endo-STTN: realce por min/max, CIE-XYZ,
    reflejo donde Y >= Y/(X+Y+Z), dilatación de 1 px."""
    img = rgb.astype(np.float64) / 255
    mn, mx = img.min(-1), img.max(-1)
    s = np.divide(mn, mx, out=np.zeros_like(mn), where=mx > 0)
    xyz = _srgb_a_xyz(s[..., None] * img)
    suma = xyz.sum(-1)
    y = np.divide(xyz[..., 1], suma, out=np.full_like(suma, np.inf), where=suma > 0)
    return _dilatar_1px(xyz[..., 1] >= y)


def mascara_arnold(rgb, t_abs=240, t_cand=200, t_rel=1.2, mediana=31):
    """Arnold et al. (2010), versión simplificada: detector absoluto (muy brillante, corregido por el balance
    de color) + detector relativo (más brillante que una imagen "sin reflejos" estimada), dilatación de 1 px."""
    img = rgb.astype(np.float32)
    gris = img @ np.array([0.2989, 0.5870, 0.1140], np.float32)
    p95 = lambda c: max(float(np.percentile(c, 95)), 1.0)
    r_g, r_b = p95(img[..., 1]) / p95(gris), p95(img[..., 2]) / p95(gris)
    sobre = lambda t: (img[..., 1] > r_g * t) | (img[..., 2] > r_b * t) | (gris > t)
    candidatos = cv2.dilate(sobre(t_cand).astype(np.uint8), cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7)))
    sin_reflejos = cv2.medianBlur(cv2.inpaint(rgb, candidatos, 5, cv2.INPAINT_TELEA), mediana).astype(np.float32)
    relativo = ((img / (sin_reflejos + 1)).max(-1) > t_rel) & (gris > t_cand * 0.75)
    return _dilatar_1px(sobre(t_abs) | relativo)


METODOS = {"meslouhi": mascara_meslouhi, "arnold": mascara_arnold}


def mascara(rgb_320, metodo):
    """Máscara a 320x256 calculada sobre el frame a 288x288, como al reentrenar Endo-STTN."""
    m = METODOS[metodo](cv2.resize(rgb_320, (LADO, LADO), interpolation=cv2.INTER_AREA))
    return cv2.resize(m.astype(np.uint8), (ANCHO, ALTO), interpolation=cv2.INTER_NEAREST).astype(bool)

In [ ]:
import warnings
import torch
import torch.nn.functional as F
warnings.filterwarnings("ignore", category=UserWarning, module="torchvision")   # aviso de 'pretrained' en AF-SfMLearner
import networks                                  # AF-SfMLearner
from model.sttn import InpaintGenerator          # Endo-STTN

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PROF_MIN, PROF_MAX = 0.1, PROF_MAX_MM


def cargar_estimador(carpeta):
    enc = networks.ResnetEncoder(18, False)
    sd = torch.load(Path(carpeta) / "encoder.pth", map_location=DEVICE)
    enc.load_state_dict({k: v for k, v in sd.items() if k in enc.state_dict()})
    dec = networks.DepthDecoder(num_ch_enc=enc.num_ch_enc, scales=range(4))
    dec.load_state_dict(torch.load(Path(carpeta) / "depth.pth", map_location=DEVICE))
    return enc.to(DEVICE), dec.to(DEVICE)


def guardar_estimador(enc, dec, carpeta):
    carpeta.mkdir(parents=True, exist_ok=True)
    sd = enc.state_dict()
    sd.update(height=ALTO, width=ANCHO)
    torch.save(sd, carpeta / "encoder.pth")
    torch.save(dec.state_dict(), carpeta / "depth.pth")


def a_tensor(rgb_uint8):
    """(B,H,W,3) uint8 -> (B,3,H,W) en [0,1]."""
    return torch.from_numpy(np.array(rgb_uint8)).permute(0, 3, 1, 2).float().div(255).to(DEVICE)


def predecir(enc, dec, x):
    """Profundidad en mm, (B,1,H,W). Igual que disp_to_depth de Monodepth2/AF-SfMLearner."""
    disp = dec(enc(x))[("disp", 0)]
    min_disp, max_disp = 1 / PROF_MAX, 1 / PROF_MIN
    return 1 / (min_disp + (max_disp - min_disp) * disp)


def perdida_si_log(pred, gt):
    """Scale-invariant log por imagen, sobre los píxeles con profundidad real."""
    perdidas = []
    for p, g in zip(pred, gt):
        valido = g > 0
        if valido.sum() < 10:
            continue
        d = torch.log(p[valido]) - torch.log(g[valido])
        perdidas.append((d ** 2).mean() - d.mean() ** 2)
    return torch.stack(perdidas).mean()


def metricas(pred, gt, region=None):
    """Métricas con median scaling. La escala se calcula con toda la imagen y luego se mide en la región."""
    valido = (gt > 0) & (gt <= PROF_MAX)
    if valido.sum() < 10:
        return None
    escala = np.median(gt[valido]) / np.median(pred[valido])
    if region is not None:
        valido &= region
        if valido.sum() < 10:
            return None
    p, g = pred[valido].astype(np.float64), gt[valido].astype(np.float64)
    p = np.clip(p * escala, PROF_MIN, PROF_MAX)
    ratio = np.maximum(p / g, g / p)
    return {"abs_rel": float(np.mean(np.abs(p - g) / g)), "sq_rel": float(np.mean((p - g) ** 2 / g)),
            "rmse": float(np.sqrt(np.mean((p - g) ** 2))),
            "rmse_log": float(np.sqrt(np.mean((np.log(p) - np.log(g)) ** 2))), "a1": float(np.mean(ratio < 1.25))}


@torch.no_grad()
def predecir_video(enc, dec, rgb):
    enc.eval(); dec.eval()
    return np.concatenate([predecir(enc, dec, a_tensor(rgb[i:i + 16]))[:, 0].cpu().numpy()
                           for i in range(0, len(rgb), 16)])

In [ ]:
NEIGHBOR_STRIDE, REF_LENGTH = 5, 10   # como en Endo-STTN/test.py
DESPLAZAMIENTO = 56


def cargar_endo(pesos):
    sd = torch.load(pesos, map_location=DEVICE)
    sd = {k.removeprefix("module."): v for k, v in sd.get("netG", sd).items()}
    modelo = InpaintGenerator().to(DEVICE)
    modelo.load_state_dict(sd)
    return modelo.eval()


@torch.no_grad()
def limpiar_bloque(modelo, rgb, mask):
    """rgb (T,H,W,3) uint8, mask (T,H,W) bool -> (T,H,W,3) uint8 con los reflejos rellenados."""
    T, H, W = mask.shape
    f = np.stack([cv2.resize(x, (LADO, LADO), interpolation=cv2.INTER_AREA) for x in rgb])
    m = np.stack([cv2.resize(x.astype(np.uint8), (LADO, LADO), interpolation=cv2.INTER_NEAREST) for x in mask])
    feats = torch.from_numpy(f).permute(0, 3, 1, 2).float().div(255).mul(2).sub(1).to(DEVICE)   # [-1,1]
    masks = torch.from_numpy(m).float().unsqueeze(1).to(DEVICE)
    enc = modelo.encoder(feats * (1 - masks))
    _, c, fh, fw = enc.shape
    enc = enc.view(1, T, c, fh, fw)
    acum, cuenta = np.zeros((T, LADO, LADO, 3), np.float32), np.zeros(T, np.float32)
    for k in range(0, T, NEIGHBOR_STRIDE):
        vec = list(range(max(0, k - NEIGHBOR_STRIDE), min(T, k + NEIGHBOR_STRIDE + 1)))
        ids = vec + [i for i in range(0, T, REF_LENGTH) if i not in vec]
        pred = torch.tanh(modelo.decoder(modelo.infer(enc[0, ids], masks[ids])[:len(vec)]))
        pred = ((pred + 1) / 2).clamp(0, 1).permute(0, 2, 3, 1).cpu().numpy() * 255
        for i, idx in enumerate(vec):
            acum[idx] += pred[i]
            cuenta[idx] += 1
    pred = acum / cuenta[:, None, None, None]
    out = rgb.copy()
    for t in range(T):
        relleno = cv2.resize(pred[t], (W, H), interpolation=cv2.INTER_LINEAR)
        out[t][mask[t]] = np.clip(relleno[mask[t]], 0, 255).astype(np.uint8)
    return out


def limpiar_video(modelo, rgb, mask):
    rgb, mask = np.asarray(rgb), np.asarray(mask)
    return np.concatenate([limpiar_bloque(modelo, rgb[i:i + BLOQUE], mask[i:i + BLOQUE])
                           for i in range(0, len(rgb), BLOQUE)])


def desplazar(m, dx=DESPLAZAMIENTO):
    """Máscara desplazada dx px a la derecha."""
    out = torch.zeros_like(m)
    out[..., dx:] = m[..., :-dx]
    return out


def rellenar(sttn, x, hueco):
    """Relleno diferenciable. x (B,T,3,H,W) en [0,1], hueco (B,T,1,H,W) -> (B,T,3,H,W)."""
    B, T, _, H, W = x.shape
    f = F.interpolate(x.flatten(0, 1), (LADO, LADO), mode="area") * 2 - 1
    m = F.interpolate(hueco.flatten(0, 1), (LADO, LADO), mode="nearest")
    out = sttn((f * (1 - m)).view(B, T, 3, LADO, LADO), m.view(B, T, 1, LADO, LADO))
    out = F.interpolate((out + 1) / 2, (H, W), mode="bilinear", align_corners=False)
    return out.view(B, T, 3, H, W)


def l1_en(a, b, region):
    """L1 media dentro de una región (B,T,1,H,W); 0 si la región está vacía."""
    return ((a - b).abs() * region).sum() / (region.sum() * a.shape[2]).clamp(min=1)

In [ ]:
def ruta(video, nombre):
    return DATOS / video / f"{nombre}.npy"


def existe(video, nombre):
    """¿Ya existe? Si solo está en Drive (de otra sesión), lo copia a local."""
    en_drive = SALIDA / "datos" / video / f"{nombre}.npy"
    if not ruta(video, nombre).exists() and en_drive.exists():
        ruta(video, nombre).parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(en_drive, ruta(video, nombre))
    return ruta(video, nombre).exists()


def guardar(video, nombre, arr):
    """Guarda en local y en Drive (primero con otro nombre, para no dejar archivos a medias)."""
    ruta(video, nombre).parent.mkdir(parents=True, exist_ok=True)
    np.save(ruta(video, nombre), arr)
    destino = SALIDA / "datos" / video
    destino.mkdir(parents=True, exist_ok=True)
    shutil.copy(ruta(video, nombre), destino / f"{nombre}.tmp")
    os.replace(destino / f"{nombre}.tmp", destino / f"{nombre}.npy")


def cargar(video, nombre):
    assert existe(video, nombre), f"Falta {nombre}.npy de {video}"
    return np.load(ruta(video, nombre), mmap_mode="r")


class Frames(torch.utils.data.Dataset):
    def __init__(self, videos, entrada):
        self.datos = [(cargar(v, entrada), cargar(v, "depth")) for v in videos]
        self.indices = [(k, i) for k, (rgb, _) in enumerate(self.datos) for i in range(len(rgb))]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, n):
        k, i = self.indices[n]
        rgb = np.array(self.datos[k][0][i], np.float32) / 255
        prof = np.array(self.datos[k][1][i], np.float32)
        if random.random() < 0.5:
            rgb, prof = rgb[:, ::-1], prof[:, ::-1]
        rgb = np.clip(rgb * random.uniform(0.9, 1.1) + random.uniform(-0.05, 0.05), 0, 1)
        return torch.from_numpy(rgb.transpose(2, 0, 1).copy()), torch.from_numpy(prof.copy())[None]


class Clips(torch.utils.data.Dataset):
    def __init__(self, videos, mascara, T=5):
        self.datos = [(cargar(v, "rgb"), cargar(v, "depth"), cargar(v, f"mask_{mascara}"),
                       cargar(v, f"rgb_limpio_{mascara}")) for v in videos]
        self.T = T
        self.indices = [(k, i) for k, d in enumerate(self.datos) for i in range(0, len(d[0]) - T + 1, T)]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, n):
        k, i = self.indices[n]
        rgb, prof, mask, limpio = self.datos[k]
        ids = sorted(random.sample(range(len(rgb)), self.T)) if random.random() < 0.5 else list(range(i, i + self.T))
        x = np.stack([rgb[j] for j in ids]).astype(np.float32) / 255
        lim = np.stack([limpio[j] for j in ids]).astype(np.float32) / 255
        m = np.stack([mask[j] for j in ids]).astype(np.float32)
        g = np.stack([prof[j] for j in ids]).astype(np.float32)
        if random.random() < 0.5:
            x, lim, m, g = x[:, :, ::-1], lim[:, :, ::-1], m[:, :, ::-1], g[:, :, ::-1]
        brillo = np.array([random.uniform(0.9, 1.1), random.uniform(-0.05, 0.05)], np.float32)
        t = lambda a: torch.from_numpy(np.ascontiguousarray(a))
        return t(x.transpose(0, 3, 1, 2)), t(lim.transpose(0, 3, 1, 2)), t(m[:, None]), t(g[:, None]), t(brillo)


def fijar_semilla(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    return torch.Generator().manual_seed(s)


print("Dispositivo:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu (sin GPU: será muy lento)")

## 2. Datos

Se reutilizan los datos que preparó el Avance 5 en Drive (`end_to_end_mascaras/datos/`). Si faltan, esta celda los
prepara con la misma configuración. Es la misma celda del Avance 5.

In [ ]:
VIDEOS = [v for g in DIVISION.values() for v in g]
prep = {"paso": PASO, "max_frames": MAX_FRAMES, "division": DIVISION}
f_prep = SALIDA / "preparacion.json"
if f_prep.exists():
    assert json.load(open(f_prep)) == prep, (
        f"Los datos de {SALIDA} se prepararon con otra configuración: {json.load(open(f_prep))}. "
        "Regresa PASO, MAX_FRAMES y DIVISION a esos valores, o usa otra carpeta SALIDA.")
else:
    json.dump(prep, open(f_prep, "w"), indent=2)


def zip_de(video):
    """ZIP de un video en CARPETA_RAW: nombre exacto, o con sufijo de Drive ("-007", " (1)")."""
    if (CARPETA_RAW / f"{video}.zip").exists():
        return CARPETA_RAW / f"{video}.zip"
    candidatos = [z for z in CARPETA_RAW.glob(f"{video}*.zip") if not z.stem[len(video)].isalnum()
                  and z.stem[len(video)] != "_"]
    assert len(candidatos) == 1, f"Para {video} hay {len(candidatos)} ZIPs en {CARPETA_RAW}: {candidatos}"
    return candidatos[0]


for k, v in enumerate(VIDEOS, 1):
    if all(existe(v, n) for n in ["rgb", "depth", "mask_meslouhi", "mask_arnold"]):
        print(f"[{k}/{len(VIDEOS)}] {v}: ya preparado")
        continue
    with zipfile.ZipFile(zip_de(v)) as zf:
        pares = listar_pares(zf)[::PASO][:MAX_FRAMES or None]
        rgbs, profs = [], []
        for _, r, d in pares:
            rgbs.append(cv2.resize(leer_rgb(zf, r), (ANCHO, ALTO), interpolation=cv2.INTER_AREA))
            profs.append(cv2.resize(leer_profundidad(zf, d), (ANCHO, ALTO),
                                    interpolation=cv2.INTER_NEAREST).astype(np.float16))
    rgb = np.stack(rgbs)
    guardar(v, "rgb", rgb)
    guardar(v, "depth", np.stack(profs))
    for m in METODOS:
        guardar(v, f"mask_{m}", np.stack([mascara(x, m) for x in rgb]))
    print(f"[{k}/{len(VIDEOS)}] {v}: {len(rgb)} frames")

import pandas as pd
from IPython.display import Markdown, display
grupo = {v: g for g, vs in DIVISION.items() for v in vs}
display(pd.DataFrame([{"video": v, "grupo": grupo[v], "frames": len(cargar(v, "rgb")),
                       **{f"% reflejos {m}": round(100 * float(np.mean(cargar(v, f"mask_{m}"))), 2) for m in METODOS}}
                      for v in VIDEOS]).set_index("video"))

## 3. Nota 4 · Auditoría de fuga de datos

Endo-STTN se reentrenó en el Avance 4 con **todos los frames** de sus videos de entrenamiento y validación, y el
estimador de profundidad se entrena y evalúa con **1 de cada 5 frames** de su propia división. La regla es que **ningún
frame de prueba de profundidad** haya pasado por el entrenamiento o la validación de Endo-STTN ni del estimador.

Se revisa en tres niveles:
1. **Video:** los nombres de los videos de prueba no aparecen en ningún conjunto de entrenamiento o validación. Los
   nombres se normalizan, así que `c1_ascending_t3_v1-007` cuenta como `c1_ascending_t3_v1`.
2. **Segmento:** tampoco otra versión del mismo segmento (`v1`, `v2`, `v3` recorren el mismo tejido).
3. **Frame:** se compara el **contenido** de cada frame, no su nombre:
   - **idéntico:** huella MD5 del frame ya preparado a 320×256. Detecta el mismo frame aunque venga de un ZIP con
     otro nombre;
   - **casi idéntico:** huella perceptual (*dHash* de 64 bits) con distancia de Hamming ≤ 2. Detecta frames
     prácticamente iguales; estos se muestran para revisarlos a ojo, porque dos frames distintos de tejido parecido
     pueden coincidir.

La división de Endo-STTN se lee del `config.json` guardado junto a sus pesos. Si no existe (la corrida 4 no lo tiene),
se usa `DIVISION_ENDO_RESPALDO` y se avisa.

In [ ]:
import pandas as pd
from IPython.display import Markdown, display

base_video = lambda v: (re.match(r"(c\d+_[a-z0-9]+_t\d+_v\d+)", v) or [v])[0] if isinstance(v, str) else v
segmento = lambda v: re.sub(r"_v\d+$", "", base_video(v))

DIVISIONES = {"profundidad": json.load(open(SALIDA / "preparacion.json"))["division"]}
for m, pesos in PESOS_ENDO.items():
    cfg = pesos.parent / "config.json"
    if cfg.exists() and "division" in json.load(open(cfg)):
        DIVISIONES[f"Endo-STTN {m}"] = json.load(open(cfg))["division"]
        print(f"Endo-STTN {m}: división leída de {cfg}")
    else:
        DIVISIONES[f"Endo-STTN {m}"] = DIVISION_ENDO_RESPALDO
        print(f"AVISO Endo-STTN {m}: no hay config.json con la división junto a {pesos.name}; "
              "se usa DIVISION_ENDO_RESPALDO (la del Avance 4). Confírmala con el notebook de esa corrida.")

prueba = {base_video(v) for v in DIVISIONES["profundidad"]["prueba"]}
filas = []
for nombre, div in DIVISIONES.items():
    for grupo in ("entrenamiento", "validacion"):
        vids = {base_video(v) for v in div[grupo]}
        filas.append({"conjunto": f"{nombre} · {grupo}", "videos": len(vids),
                      "mismo video que prueba": ", ".join(sorted(vids & prueba)) or "ninguno",
                      "mismo segmento que prueba": ", ".join(sorted(v for v in vids if segmento(v) in {segmento(p) for p in prueba})) or "ninguno"})
NIVEL_12 = pd.DataFrame(filas)
display(Markdown("**Niveles 1 y 2: videos y segmentos de prueba frente a cada conjunto de entrenamiento y validación**"))
display(NIVEL_12.set_index("conjunto"))

In [ ]:
def dhash(img, lado=8):
    """Huella perceptual de 64 bits: compara brillo de píxeles vecinos en una miniatura de 9x8."""
    g = cv2.resize(cv2.cvtColor(img, cv2.COLOR_RGB2GRAY), (lado + 1, lado), interpolation=cv2.INTER_AREA)
    return np.packbits((g[:, 1:] > g[:, :-1]).ravel()).tobytes()


def huellas_zip(video, paso):
    """MD5 y dHash de los frames de un video leídos de su ZIP, preparados a 320x256 como en el Avance 5."""
    with zipfile.ZipFile(zip_de(base_video(video))) as zf:
        rutas = sorted((n for n in zf.namelist() if n.startswith("rgb/") and n.lower().endswith((".png", ".jpg"))),
                       key=lambda n: (indice(n) is None, indice(n) or 0, n))[::paso]
        for r in rutas:
            f = cv2.resize(leer_rgb(zf, r), (ANCHO, ALTO), interpolation=cv2.INTER_AREA)
            yield hashlib.md5(f.tobytes()).hexdigest(), dhash(f), r


# Frames de prueba de profundidad: los que de verdad se evalúan (rgb.npy del Avance 5)
PRUEBA_FRAMES = []
for v in DIVISIONES["profundidad"]["prueba"]:
    rgb = cargar(v, "rgb")
    PRUEBA_FRAMES += [(v, i, hashlib.md5(np.ascontiguousarray(rgb[i]).tobytes()).hexdigest(), dhash(np.asarray(rgb[i])))
                      for i in range(len(rgb))]
md5_prueba = {h: (v, i) for v, i, h, _ in PRUEBA_FRAMES}
dh_prueba = np.array([np.frombuffer(d, np.uint8) for *_, d in PRUEBA_FRAMES])

# Frames que vio cada entrenamiento: Endo-STTN con todos los frames, el estimador con 1 de cada PASO
a_revisar, filas = [], []
for nombre, div in DIVISIONES.items():
    paso = PASO_ENDO if nombre.startswith("Endo") else PASO
    for grupo in ("entrenamiento", "validacion"):
        n = identicos = casi = 0
        for v in div[grupo]:
            for h, d, r in huellas_zip(v, paso):
                n += 1
                if h in md5_prueba:
                    identicos += 1
                    a_revisar.append((nombre, grupo, v, r, *md5_prueba[h], "idéntico"))
                    continue
                dist = np.unpackbits(dh_prueba ^ np.frombuffer(d, np.uint8), axis=1).sum(1)
                if dist.min() <= 2:
                    casi += 1
                    j = int(dist.argmin())
                    a_revisar.append((nombre, grupo, v, r, PRUEBA_FRAMES[j][0], PRUEBA_FRAMES[j][1], f"casi idéntico ({dist.min()} bits)"))
        filas.append({"conjunto": f"{nombre} · {grupo}", "frames revisados": n,
                      "idénticos a un frame de prueba": identicos, "casi idénticos": casi})
        print(f"{nombre} · {grupo}: {n} frames revisados", flush=True)

NIVEL_3 = pd.DataFrame(filas)
display(Markdown(f"**Nivel 3: {len(PRUEBA_FRAMES)} frames de prueba frente a los frames de cada conjunto**"))
display(NIVEL_3.set_index("conjunto"))
pd.DataFrame(a_revisar, columns=["conjunto", "grupo", "video", "frame", "video prueba", "frame prueba", "tipo"]) \
    .to_csv(RES / "fuga_frames_a_revisar.csv", index=False)

cruces_video = (NIVEL_12["mismo video que prueba"] != "ninguno").sum()
cruces_segmento = (NIVEL_12["mismo segmento que prueba"] != "ninguno").sum()
identicos = int(NIVEL_3["idénticos a un frame de prueba"].sum())
casi = int(NIVEL_3["casi idénticos"].sum())
with open(RES / "auditoria_fuga.md", "w") as f:
    f.write("# Auditoría de fuga de datos\n\n" + NIVEL_12.to_markdown(index=False) + "\n\n" + NIVEL_3.to_markdown(index=False) + "\n")
print(f"\nResultado: {cruces_video} cruces de video, {cruces_segmento} de segmento, {identicos} frames idénticos, "
      f"{casi} casi idénticos (ver fuga_frames_a_revisar.csv).")
assert cruces_video == 0 and identicos == 0, "HAY FUGA DE DATOS: los resultados de prueba no son válidos"
if cruces_segmento or casi:
    print("AVISO: revisa los cruces de segmento y los frames casi idénticos antes de reportar resultados.")
else:
    print("Sin fuga: ningún frame de prueba se usó para entrenar ni validar Endo-STTN ni el estimador.")

In [ ]:
import matplotlib.pyplot as plt

# Frames casi idénticos (si los hay): el de entrenamiento junto al de prueba, para decidir a ojo
casos = [c for c in a_revisar if c[-1].startswith("casi")][:6]
if casos:
    fig, ax = plt.subplots(len(casos), 2, figsize=(8, 3.2 * len(casos)), squeeze=False)
    for fila, (nombre, grupo, v, r, vp, ip, tipo) in zip(ax, casos):
        with zipfile.ZipFile(zip_de(base_video(v))) as zf:
            fila[0].imshow(cv2.resize(leer_rgb(zf, r), (ANCHO, ALTO), interpolation=cv2.INTER_AREA))
        fila[1].imshow(cargar(vp, "rgb")[ip])
        fila[0].set_title(f"{nombre} · {grupo}\n{v} · {Path(r).name}", fontsize=8)
        fila[1].set_title(f"prueba · {vp} · frame {ip}\n{tipo}", fontsize=8)
        for a in fila: a.axis("off")
    plt.tight_layout(); plt.savefig(RES / "fuga_casi_identicos.png", dpi=80); plt.show()
else:
    print("No hay frames casi idénticos que revisar.")

## 4. Nota 3 · El estimador solo

AF-SfMLearner **tal cual**: pesos SCARED, frames originales, sin limpiar los reflejos ni ajustarlo a C3VDv2 (el
modelo A). Muestra el punto de partida antes de cualquier ajuste. Las cifras con intervalos de confianza están en la
sección 6.

In [ ]:
VIDEOS_PRUEBA = DIVISIONES["profundidad"]["prueba"]
UNION = {v: np.asarray(cargar(v, "mask_meslouhi")) | np.asarray(cargar(v, "mask_arnold")) for v in VIDEOS_PRUEBA}

enc, dec = cargar_estimador(PESOS_SCARED)
pred_A = {v: predecir_video(enc, dec, cargar(v, "rgb")) for v in VIDEOS_PRUEBA}
del enc, dec

v = VIDEOS_PRUEBA[0]
rgb, prof = cargar(v, "rgb"), cargar(v, "depth")
elegidos = np.linspace(0, len(rgb) - 1, 4).astype(int)
fig, ax = plt.subplots(4, len(elegidos), figsize=(4 * len(elegidos), 12), squeeze=False)
for j, i in enumerate(elegidos):
    g = np.asarray(prof[i], np.float32); valido = g > 0
    p = pred_A[v][i] * np.median(g[valido]) / np.median(pred_A[v][i][valido])
    err = np.where(valido, np.abs(p - g) / np.maximum(g, 1e-3), np.nan)
    vmin, vmax = np.percentile(g[valido], [2, 98])
    paneles = [(rgb[i], {}, f"{v} · frame {i}"), (np.where(valido, g, np.nan), dict(cmap="magma_r", vmin=vmin, vmax=vmax), "profundidad real"),
               (p, dict(cmap="magma_r", vmin=vmin, vmax=vmax), "estimador solo (A)"),
               (err, dict(cmap="inferno", vmin=0, vmax=0.5), f"error relativo (media {np.nanmean(err):.3f})")]
    for r, (img, kw, t) in enumerate(paneles):
        ax[r, j].imshow(img, **kw); ax[r, j].set_title(t, fontsize=9); ax[r, j].axis("off")
    ax[3, j].contour(UNION[v][i], [0.5], colors="cyan", linewidths=0.5)
plt.suptitle("Estimador sin ajustar sobre frames originales (contorno cian = reflejos)", y=1.0)
plt.tight_layout(); plt.savefig(RES / "estimador_solo.png", dpi=80, bbox_inches="tight"); plt.show()

## 5. Nota 2 · Endo-STTN solo, sin profundidad

Se entrena **solo el inpainting**, con cada máscara, partiendo del Endo-STTN reentrenado del Avance 4. Es **el mismo
entrenamiento que el modelo E** (clips de 5 frames, lote, épocas, tasa de aprendizaje, semillas y la pérdida de hueco
$\mathcal{L}_{hueco}$), pero **sin la pérdida de profundidad** y sin estimador. Así se separa qué aporta seguir
entrenando el relleno y qué aporta la señal de profundidad.

Después se hace la inferencia en los videos de prueba y se mide la calidad del relleno en el **pseudo-hueco**: la
máscara de reflejos desplazada sobre tejido sano, donde se conoce el color real. Se comparan cinco rellenos:

| Relleno | Origen |
|---|---|
| Original | Endo-STTN publicado (`gen_00009.pth`), si está disponible |
| Avance 4 (congelado) | El que usa el modelo D |
| **Endo solo** | Este entrenamiento, sin profundidad |
| E | El Endo-STTN ajustado junto con el estimador (`sttn.pth` de E) |

Se mide con los huecos de **la propia máscara** de cada modelo y, para comparar entre máscaras, con los huecos de
**El Meslouhi** para todos (la misma regla que en el Avance 4).

In [ ]:
from skimage.metrics import structural_similarity


def hueco_np(mask):
    """Pseudo-hueco a 320x256: la máscara desplazada DESPLAZAMIENTO px, sin los reflejos reales."""
    h = np.zeros_like(mask)
    h[..., DESPLAZAMIENTO:] = mask[..., :-DESPLAZAMIENTO]
    return h & ~mask


def metricas_hueco(pred, real, hueco, minimo=30):
    """MAE y PSNR en los píxeles del hueco; SSIM de la imagen promediado en el hueco (como el Avance 4)."""
    if hueco.sum() < minimo:
        return None
    d = pred[hueco].astype(np.float64) - real[hueco].astype(np.float64)
    _, mapa = structural_similarity(real, pred, channel_axis=2, data_range=255, full=True)
    return {"mae": float(np.abs(d).mean()), "psnr": float(10 * np.log10(255 ** 2 / max(float((d ** 2).mean()), 1e-10))),
            "ssim": float(mapa.mean(-1)[hueco].mean())}


class ClipsEndo(torch.utils.data.Dataset):
    """Clips de 5 frames con su máscara, como Clips del Avance 5 pero sin profundidad."""
    def __init__(self, videos, mascara, T=5):
        self.datos = [(cargar(v, "rgb"), cargar(v, f"mask_{mascara}")) for v in videos]
        self.T = T
        self.indices = [(k, i) for k, (r, _) in enumerate(self.datos) for i in range(0, len(r) - T + 1, T)]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, n):
        k, i = self.indices[n]
        rgb, mask = self.datos[k]
        ids = sorted(random.sample(range(len(rgb)), self.T)) if random.random() < 0.5 else list(range(i, i + self.T))
        x = np.stack([rgb[j] for j in ids]).astype(np.float32) / 255
        m = np.stack([mask[j] for j in ids]).astype(np.float32)
        if random.random() < 0.5:
            x, m = x[:, :, ::-1], m[:, :, ::-1]
        t = lambda a: torch.from_numpy(np.ascontiguousarray(a))
        return t(x.transpose(0, 3, 1, 2)), t(m[:, None])


def evaluar_relleno(sttn, videos, mascara_huecos):
    """Métricas por frame del relleno en el pseudo-hueco (inferencia oficial de Endo-STTN)."""
    filas = []
    for v in videos:
        rgb, mask = np.asarray(cargar(v, "rgb")), np.asarray(cargar(v, f"mask_{mascara_huecos}"))
        hueco = hueco_np(mask)
        lleno = limpiar_video(sttn, rgb, hueco)
        for i in range(len(rgb)):
            if (r := metricas_hueco(lleno[i], rgb[i], hueco[i])):
                filas.append({"video": v, "frame": i, **r})
    return filas


def entrenar_endo_solo(mascara, semilla):
    nombre = f"EndoSolo_{mascara}_s{semilla}"
    salida = SALIDA_PRUEBAS / "modelos" / nombre
    if (salida / "historial.json").exists() and "mejor_mae_val" in json.load(open(salida / "historial.json")):
        print(f"{nombre}: ya entrenado, se reutiliza"); return salida
    salida.mkdir(parents=True, exist_ok=True)
    gen = fijar_semilla(semilla)
    datos = ClipsEndo(DIVISION["entrenamiento"], mascara)
    carga = torch.utils.data.DataLoader(datos, batch_size=LOTE_ENDO, shuffle=True, drop_last=True, generator=gen)
    sttn = cargar_endo(PESOS_ENDO[mascara])
    opt = torch.optim.Adam(sttn.parameters(), lr=LR_STTN, betas=(0.0, 0.99))
    valida = lambda: float(np.mean([f["mae"] for f in evaluar_relleno(sttn.eval(), DIVISION["validacion"], mascara)]))
    historial = {"mascara": mascara, "semilla": semilla, "epocas": [{"epoca": 0, "mae_val": valida()}]}
    mejor = historial["epocas"][0]["mae_val"]
    torch.save({"netG": sttn.state_dict()}, salida / "sttn.pth")
    print(f"{nombre} ({len(datos)} clips) · época 0: MAE validación {mejor:.3f}")
    for ep in range(1, EPOCAS_ENDO + 1):
        sttn.train(); t0, perdidas = time.time(), []
        for x, m in carga:
            x, m = x.to(DEVICE), m.to(DEVICE)
            pseudo = desplazar(m) * (1 - m)
            relleno = rellenar(sttn, x, (m + pseudo).clamp(max=1))
            loss = l1_en(relleno, x, pseudo)          # solo la pérdida de hueco: sin profundidad
            opt.zero_grad(); loss.backward(); opt.step()
            perdidas.append(loss.item())
        val = valida()
        historial["epocas"].append({"epoca": ep, "perdida_hueco": float(np.mean(perdidas)), "mae_val": val})
        marca = ""
        if val < mejor:
            mejor, marca = val, "  <- mejor, guardado"
            torch.save({"netG": sttn.state_dict()}, salida / "sttn.pth")
        print(f"  época {ep}: pérdida hueco {np.mean(perdidas):.4f} | MAE validación {val:.3f} | {time.time() - t0:.0f} s{marca}")
        json.dump(historial, open(salida / "historial.json", "w"), indent=2)
    historial["mejor_mae_val"] = mejor
    json.dump(historial, open(salida / "historial.json", "w"), indent=2)
    del sttn, opt; torch.cuda.empty_cache()
    return salida


ENDO_SOLO = {(m, s): entrenar_endo_solo(m, s) for m in PESOS_ENDO for s in SEMILLAS}

In [ ]:
def buscar(nombre, carpetas):
    """Primera carpeta de modelo que existe (con historial terminado) entre las carpetas dadas."""
    for c in carpetas:
        f = Path(c) / nombre / "historial.json"
        if f.exists() and "mejor_abs_rel_val" in json.load(open(f)):
            return Path(c) / nombre
    return None


filas = []
for m in PESOS_ENDO:
    rellenos = {"Original": [PESOS_ORIGINAL] if INCLUIR_ORIGINAL else [],
                "Avance 4 (congelado)": [PESOS_ENDO[m]],
                "Endo solo": [ENDO_SOLO[(m, s)] / "sttn.pth" for s in SEMILLAS],
                "E": [c / "sttn.pth" for s in SEMILLAS if (c := buscar(f"E_{m}_s{s}", CARPETAS_PRINCIPALES))]}
    for nombre, lista in rellenos.items():
        for k, pesos in enumerate(lista):
            sttn = cargar_endo(pesos)
            for huecos in dict.fromkeys([m, "meslouhi"]):
                for f in evaluar_relleno(sttn, VIDEOS_PRUEBA, huecos):
                    filas.append({"máscara": m, "relleno": nombre, "corrida": k, "huecos": huecos, **f})
            del sttn; torch.cuda.empty_cache()
        print(f"{m} · {nombre}: {len(lista)} modelo(s) evaluado(s)", flush=True)

RELLENO = pd.DataFrame(filas)
RELLENO.to_csv(RES / "relleno_por_frame.csv", index=False)
orden = ["Original", "Avance 4 (congelado)", "Endo solo", "E"]
tabla = (RELLENO.groupby(["huecos", "máscara", "relleno", "corrida"])[["psnr", "ssim", "mae"]].mean()
         .groupby(["huecos", "máscara", "relleno"]).agg(["mean", "std"]).round(3))
tabla.to_csv(RES / "relleno_resumen.csv")
for huecos in tabla.index.get_level_values(0).unique():
    display(Markdown(f"**Calidad del relleno en prueba · huecos de {huecos}** (media y desviación entre semillas)"))
    t = tabla.loc[huecos]
    display(t.reindex([(m, r) for m in PESOS_ENDO for r in orden if (m, r) in t.index]))

In [ ]:
# Relleno en el frame de prueba con más hueco: respuesta conocida (hueco) y reflejo real
v = VIDEOS_PRUEBA[0]
rgb = np.asarray(cargar(v, "rgb"))
for m in PESOS_ENDO:
    mask = np.asarray(cargar(v, f"mask_{m}"))
    hueco = hueco_np(mask)
    i = int(np.argmax(hueco.reshape(len(hueco), -1).sum(1)))
    seg = slice(max(0, i - 10), min(len(rgb), i + 11)); j = i - seg.start
    rellenos = {"Original": PESOS_ORIGINAL if INCLUIR_ORIGINAL else None, "Avance 4": PESOS_ENDO[m],
                "Endo solo": ENDO_SOLO[(m, SEMILLAS[0])] / "sttn.pth",
                "E": (c / "sttn.pth") if (c := buscar(f"E_{m}_s{SEMILLAS[0]}", CARPETAS_PRINCIPALES)) else None}
    rellenos = {k: p for k, p in rellenos.items() if p is not None}

    def zoom(mk, lado=96):
        ys, xs = np.nonzero(mk) if mk.any() else (np.array([ALTO // 2]), np.array([ANCHO // 2]))
        y0 = int(np.clip(np.median(ys) - lado // 2, 0, ALTO - lado)); x0 = int(np.clip(np.median(xs) - lado // 2, 0, ANCHO - lado))
        return lambda a: a[y0:y0 + lado, x0:x0 + lado]

    zh, zm = zoom(hueco[i]), zoom(mask[i])
    fig, ax = plt.subplots(2, len(rellenos) + 1, figsize=(3.4 * (len(rellenos) + 1), 6.8), squeeze=False)
    ax[0, 0].imshow(zh(rgb[i])); ax[0, 0].set_title("hueco: color real", fontsize=9)
    ax[1, 0].imshow(zm(rgb[i])); ax[1, 0].set_title("reflejo real", fontsize=9)
    for k, (nombre, pesos) in enumerate(rellenos.items(), 1):
        sttn = cargar_endo(pesos)
        h = limpiar_video(sttn, rgb[seg], hueco[seg])[j]
        r = limpiar_video(sttn, rgb[seg], mask[seg])[j]
        met = metricas_hueco(h, rgb[i], hueco[i])
        ax[0, k].imshow(zh(h)); ax[0, k].set_title(f"{nombre}: hueco\nPSNR {met['psnr']:.1f} dB · MAE {met['mae']:.1f}" if met else nombre, fontsize=9)
        ax[1, k].imshow(zm(r)); ax[1, k].set_title(f"{nombre}: reflejo rellenado", fontsize=9)
        del sttn
    for a in ax.ravel(): a.axis("off")
    plt.suptitle(f"Máscara {m} · {v} · frame {i} (zoom)", y=1.0)
    plt.tight_layout(); plt.savefig(RES / f"relleno_{m}.png", dpi=90, bbox_inches="tight"); plt.show()

## 6. Notas 1 y 5 · Prueba de profundidad con intervalos de confianza

Prueba **separada del entrenamiento**: cada modelo se reconstruye **a partir de sus pesos guardados** y se aplica a los
videos de prueba. No se reutilizan frames limpios ni resultados intermedios del Avance 5.

| Modelo | Qué se hace en la prueba |
|---|---|
| A | estimador SCARED sobre frames originales |
| C | estimador de C sobre frames originales |
| D | limpiar con el Endo-STTN congelado del Avance 4 y estimar con el estimador de D |
| E (y controles) | limpiar con el `sttn.pth` del propio E y estimar con su estimador |

**Métricas** (*median scaling* por frame, como en el Avance 5): AbsRel, Sq Rel, RMSE (mm), RMSE log y δ < 1.25, en la
imagen completa y en las zonas con reflejo (unión de las dos máscaras).

**Intervalos de confianza del 95 %:** *bootstrap* por bloques. Se remuestrean bloques de `LARGO_BLOQUE` frames
consecutivos dentro de cada video (los frames vecinos están correlacionados, así que remuestrear frames sueltos daría
intervalos demasiado estrechos), se promedian las semillas disponibles y se toman los percentiles 2.5 y 97.5 de
`N_BOOTSTRAP` repeticiones. Además se calcula el intervalo de la **diferencia pareada** E − D: si no incluye el 0, la
diferencia es significativa.

In [ ]:
def modelos_a_evaluar():
    """{nombre: [(tipo, máscara, carpeta) por semilla]} con lo que exista en Drive."""
    out = {"A · sin ajustar": [("A", None, PESOS_SCARED)]}
    c = [x for s in SEMILLAS if (x := buscar(f"C_s{s}", CARPETAS_PRINCIPALES))]
    out["C · sin limpiar"] = [("C", None, x) for x in c]
    for m in PESOS_ENDO:
        etiqueta = {"meslouhi": "Meslouhi", "arnold": "Arnold"}[m]
        for t in ("D", "E"):
            out[f"{t} · {etiqueta}"] = [(t, m, x) for s in SEMILLAS if (x := buscar(f"{t}_{m}_s{s}", CARPETAS_PRINCIPALES))]
    for nombre, carpeta in CONTROLES.items():
        out[nombre] = [("E", "meslouhi", x) for s in SEMILLAS if (x := buscar(f"E_meslouhi_s{s}", [carpeta]))]
    return {k: v for k, v in out.items() if v}


MODELOS = modelos_a_evaluar()
display(pd.DataFrame([{"modelo": k, "semillas encontradas": len(v), "carpetas": ", ".join(sorted({str(x[2].parent) for x in v}))}
                      for k, v in MODELOS.items()]).set_index("modelo"))

METRICAS = ["abs_rel", "sq_rel", "rmse", "rmse_log", "a1"]
limpios_D = {}   # el Endo-STTN congelado es el mismo para todas las semillas de D: se limpia una vez
filas = []
for nombre, corridas in MODELOS.items():
    for k, (tipo, m, carpeta) in enumerate(corridas):
        for v in VIDEOS_PRUEBA:
            rgb = cargar(v, "rgb")
            if tipo == "D":
                if (m, v) not in limpios_D:
                    sttn = cargar_endo(PESOS_ENDO[m]); limpios_D[(m, v)] = limpiar_video(sttn, rgb, cargar(v, f"mask_{m}")); del sttn
                entrada = limpios_D[(m, v)]
            elif tipo == "E":
                sttn = cargar_endo(carpeta / "sttn.pth"); entrada = limpiar_video(sttn, rgb, cargar(v, f"mask_{m}")); del sttn
            else:
                entrada = rgb
            enc, dec = cargar_estimador(carpeta)
            pred = predecir_video(enc, dec, entrada); del enc, dec
            prof = cargar(v, "depth")
            for i in range(len(pred)):
                g = np.asarray(prof[i], np.float32)
                for region, reg in (("imagen", None), ("reflejos", UNION[v][i])):
                    if (r := metricas(pred[i], g, reg)):
                        filas.append({"modelo": nombre, "corrida": k, "video": v, "frame": i, "region": region, **r})
        torch.cuda.empty_cache()
    print(f"{nombre}: {len(corridas)} semilla(s)", flush=True)

PROF = pd.DataFrame(filas)
PROF.to_csv(RES / "profundidad_por_frame.csv", index=False)

In [ ]:
rng = np.random.default_rng(0)


def muestras_bloques(n_por_video, largo, B):
    """Índices de B remuestras por bloques: en cada video se toman bloques de frames consecutivos al azar."""
    salida = []
    for _ in range(B):
        idx, base = [], 0
        for n in n_por_video:
            inicios = rng.integers(0, max(1, n - largo + 1), size=int(np.ceil(n / largo)))
            idx.append((base + (inicios[:, None] + np.arange(largo)).ravel()[:n]).clip(max=base + n - 1))
            base += n
        salida.append(np.concatenate(idx))
    return np.array(salida)


def matriz(df, modelo, region, metrica):
    """Semillas x frames (mismo orden de frames para todos los modelos); NaN si la métrica no aplica en ese frame."""
    d = df[(df.modelo == modelo) & (df.region == region)]
    t = d.pivot_table(index="corrida", columns=["video", "frame"], values=metrica)
    return t.reindex(columns=CLAVES).to_numpy()


CLAVES = pd.MultiIndex.from_tuples([(v, i) for v in VIDEOS_PRUEBA for i in range(len(cargar(v, "rgb")))], names=["video", "frame"])
IDX = muestras_bloques([len(cargar(v, "rgb")) for v in VIDEOS_PRUEBA], LARGO_BLOQUE, N_BOOTSTRAP)


def estimar(M):
    """Media (semillas y frames) e IC 95 % por bootstrap de bloques sobre frames."""
    por_frame = np.nanmean(M, axis=0)                     # promedio de semillas en cada frame
    boot = np.nanmean(por_frame[IDX], axis=1)
    return float(np.nanmean(por_frame)), *np.nanpercentile(boot, [2.5, 97.5])


filas = []
for region in ("imagen", "reflejos"):
    for modelo in MODELOS:
        fila = {"región": region, "modelo": modelo, "semillas": len(MODELOS[modelo])}
        for met in METRICAS:
            fila[met], fila[f"{met}_lo"], fila[f"{met}_hi"] = estimar(matriz(PROF, modelo, region, met))
        filas.append(fila)
IC = pd.DataFrame(filas)
IC.to_csv(RES / "profundidad_ic95.csv", index=False)

ENCABEZADO = {"abs_rel": "Abs Rel ↓", "sq_rel": "Sq Rel ↓", "rmse": "RMSE (mm) ↓", "rmse_log": "RMSE log ↓", "a1": "δ < 1.25 ↑"}
fmt = lambda r, m: f"{r[m]:.3f} [{r[m + '_lo']:.3f}, {r[m + '_hi']:.3f}]"
for region in ("imagen", "reflejos"):
    t = IC[IC["región"] == region]
    vista = pd.DataFrame({"Modelo": t.modelo, "Semillas": t.semillas, **{ENCABEZADO[m]: [fmt(r, m) for _, r in t.iterrows()] for m in METRICAS}})
    display(Markdown(f"**Prueba de profundidad · {region}** (media [IC 95 %])"))
    display(vista.set_index("Modelo"))

In [ ]:
# Diferencias pareadas: mismo remuestreo de frames para los dos modelos
COMPARACIONES = [("E · Meslouhi", "D · Meslouhi"), ("E · Meslouhi", "C · sin limpiar"), ("D · Meslouhi", "C · sin limpiar"),
                 ("E · Arnold", "D · Arnold"), ("E · Meslouhi", "E · Arnold"),
                 ("E · Meslouhi", "E · Meslouhi, STTN congelado"), ("E · Meslouhi con ancla", "D · Meslouhi")]
filas = []
for region in ("imagen", "reflejos"):
    for a, b in COMPARACIONES:
        if a not in MODELOS or b not in MODELOS:
            continue
        for met in ("abs_rel", "rmse", "a1"):
            da = np.nanmean(matriz(PROF, a, region, met), axis=0)
            db = np.nanmean(matriz(PROF, b, region, met), axis=0)
            dif = da - db
            boot = np.nanmean(dif[IDX], axis=1)
            lo, hi = np.nanpercentile(boot, [2.5, 97.5])
            mejor = "a ≠ b" if lo > 0 or hi < 0 else "sin diferencia"
            if mejor == "a ≠ b":
                mejor = f"mejor {a if (hi < 0) == (met != 'a1') else b}"
            filas.append({"región": region, "comparación": f"{a} − {b}", "métrica": ENCABEZADO[met],
                          "diferencia": round(float(np.nanmean(dif)), 4), "IC 95 %": f"[{lo:.4f}, {hi:.4f}]", "resultado": mejor})
DIFS = pd.DataFrame(filas)
DIFS.to_csv(RES / "diferencias_ic95.csv", index=False)
display(Markdown("**Diferencias pareadas con IC 95 %** (si el intervalo no incluye 0, la diferencia es significativa)"))
display(DIFS.set_index(["región", "comparación", "métrica"]))

In [ ]:
# Tabla en LaTeX lista para el artículo (imagen completa) y gráfica de intervalos
t = IC[IC["región"] == "imagen"]
lineas = [r"\begin{tabular}{l" + "c" * len(METRICAS) + "}", r"\toprule",
          "Modelo & " + " & ".join([r"Abs Rel $\downarrow$", r"Sq Rel $\downarrow$", r"RMSE (mm) $\downarrow$",
                                    r"RMSE log $\downarrow$", r"$\delta<1.25$ $\uparrow$"]) + r" \\", r"\midrule"]
for _, r in t.iterrows():
    celdas = [f"{r[m]:.3f} [{r[m + '_lo']:.3f}, {r[m + '_hi']:.3f}]" for m in METRICAS]
    lineas.append(f"{r.modelo} & " + " & ".join(celdas) + r" \\")
lineas += [r"\bottomrule", r"\end{tabular}"]
open(RES / "tabla_profundidad_ic95.tex", "w").write("\n".join(lineas) + "\n")
print("\n".join(lineas))

COLOR = {"A": "#6250d6", "C": "#2a78d6", "Meslouhi": "#eb6834", "Arnold": "#1baf7a"}
color = lambda n: COLOR["A"] if n.startswith("A") else COLOR["C"] if n.startswith("C") else COLOR["Arnold"] if "Arnold" in n else COLOR["Meslouhi"]
fig, ax = plt.subplots(1, 2, figsize=(15, 0.55 * len(MODELOS) + 1.5), sharey=True)
for a, region in zip(ax, ("imagen", "reflejos")):
    t = IC[(IC["región"] == region) & ~IC.modelo.str.startswith("A")].reset_index(drop=True)
    for k, r in t.iterrows():
        a.errorbar(r.abs_rel, k, xerr=[[r.abs_rel - r.abs_rel_lo], [r.abs_rel_hi - r.abs_rel]], fmt="o", ms=7,
                   color=color(r.modelo), capsize=4, lw=2, mfc="white" if r.modelo.startswith("D") else color(r.modelo))
    a.set_yticks(range(len(t))); a.set_yticklabels(t.modelo); a.invert_yaxis()
    a.set_title(f"AbsRel con IC 95 % · {region}"); a.set_xlabel("AbsRel (menor es mejor)"); a.grid(axis="x", alpha=0.25)
plt.tight_layout(); plt.savefig(RES / "profundidad_ic95.png", dpi=100, bbox_inches="tight"); plt.show()
print("Resultados en", RES)

## 7. Cómo leer los resultados

- **Auditoría (nota 4):** debe terminar en *Sin fuga*. Si aparecen cruces de segmento o frames casi idénticos, revisa
  `fuga_casi_identicos.png` antes de reportar. Un cruce de video o un frame idéntico detiene el notebook.
- **Estimador solo (nota 3):** es el modelo A. Su error alto (AbsRel cercano a 0.4) muestra cuánto aporta ajustar el
  estimador a C3VDv2, antes de pensar en los reflejos.
- **Endo-STTN solo (nota 2):** si *Endo solo* rellena igual o mejor que E en el pseudo-hueco, entonces la mejora de E en
  profundidad **no** viene de un mejor inpainting, sino de adaptar el relleno a lo que necesita el estimador. Si *Endo
  solo* rellena peor que E, parte de la mejora puede venir solo de seguir entrenando el relleno.
- **Prueba de profundidad (notas 1 y 5):** la tabla con IC 95 % es la que conviene reportar en el artículo
  (`tabla_profundidad_ic95.tex`). Una diferencia entre dos modelos es significativa cuando el intervalo de la diferencia
  pareada no incluye el 0.
- **Cuidado con la interpretación:** solo hay dos videos de prueba de un mismo segmento. El *bootstrap* por bloques
  corrige la correlación entre frames vecinos, pero no la falta de segmentos distintos: los intervalos describen la
  incertidumbre **dentro de este segmento**, no la generalización a otros.